# Diapilot — Experiment 3: Deep Neural Network (DNN)
### Patient-Aware Meal Scoring using Multi-Input Deep Learning
**Key advantage over KNN & XGBoost:** DNN takes BOTH patient profile AND meal nutrition as input simultaneously.
It learns that the same meal can be good for one patient and bad for another — something KNN and XGBoost cannot do.

In [1]:
%pip install tensorflow pandas scikit-learn fastparquet

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\SAAD\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# ════════════════════════════════════════════════════════
# CELL 1 — IMPORTS & SETUP
# ════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import os

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print('=' * 58)
print('   DIAPILOT — Experiment 3: Deep Neural Network')
print('=' * 58)
print(f'TensorFlow version : {tf.__version__}')
print(f'Keras version      : {keras.__version__}')

df = pd.read_parquet('../data/processed/DiaPilot_Combined_Data.parquet')
print(f'Dataset loaded     : {len(df):,} recipes')

   DIAPILOT — Experiment 3: Deep Neural Network
TensorFlow version : 2.21.0
Keras version      : 3.14.1
Dataset loaded     : 117,662 recipes


In [3]:
# ════════════════════════════════════════════════════════
# CELL 2 — MEDICAL ENGINE (same as KNN & XGBoost)
# ════════════════════════════════════════════════════════

def calculate_tdee(age, weight_kg, height_cm, gender, activity='sedentary'):
    """Mifflin-St Jeor Equation."""
    if gender.lower() == 'male':
        bmr = (10 * weight_kg) + (6.25 * height_cm) - (5 * age) + 5
    else:
        bmr = (10 * weight_kg) + (6.25 * height_cm) - (5 * age) - 161
    multipliers = {'sedentary':1.2,'light':1.375,'moderate':1.55,'active':1.725}
    return round(bmr * multipliers.get(activity, 1.2))


def get_dynamic_macros(tdee, leicester_score, on_medication=False,
                        current_glucose_high=False):
    """ADA 2025-aligned per-meal constraints."""
    if leicester_score <= 10:
        carb_pct, sugar_limit = 0.50, 15.0
    elif leicester_score <= 15:
        carb_pct, sugar_limit = 0.40, 10.0
    else:
        carb_pct, sugar_limit = 0.30,  8.0

    if current_glucose_high:
        carb_pct -= 0.10
        sugar_limit = 3.0

    per_meal_carbs_max = round(((tdee * carb_pct) / 4) / 3)
    per_meal_carbs_min = 25 if on_medication else 0
    if on_medication and per_meal_carbs_max < 25:
        per_meal_carbs_max = 25

    return {
        'per_meal_calories'  : round(tdee / 3),
        'per_meal_max_carbs' : per_meal_carbs_max,
        'per_meal_min_carbs' : per_meal_carbs_min,
        'per_meal_sugar'     : sugar_limit
    }


# ── Patient profile ─────────────────────────────────────
PATIENT_AGE       = 45
PATIENT_WEIGHT_KG = 85
PATIENT_HEIGHT_CM = 175
PATIENT_GENDER    = 'male'
PATIENT_ACTIVITY  = 'sedentary'
LEICESTER_SCORE   = 14
ON_MEDICATION     = True
GLUCOSE_HIGH      = False

patient_tdee   = calculate_tdee(PATIENT_AGE, PATIENT_WEIGHT_KG,
                                 PATIENT_HEIGHT_CM, PATIENT_GENDER,
                                 PATIENT_ACTIVITY)
patient_macros = get_dynamic_macros(patient_tdee, LEICESTER_SCORE,
                                     ON_MEDICATION, GLUCOSE_HIGH)

print('Medical Engine Ready')
print('-' * 40)
print(f'  TDEE             : {patient_tdee} kcal/day')
print(f'  Per-meal calories: {patient_macros["per_meal_calories"]} kcal')
print(f'  Carb window      : {patient_macros["per_meal_min_carbs"]}g'
      f' - {patient_macros["per_meal_max_carbs"]}g')
print(f'  Sugar max        : {patient_macros["per_meal_sugar"]}g')

Medical Engine Ready
----------------------------------------
  TDEE             : 2068 kcal/day
  Per-meal calories: 689 kcal
  Carb window      : 25g - 69g
  Sugar max        : 10.0g


In [4]:
# ════════════════════════════════════════════════════════
# CELL 3 — SYNTHETIC TRAINING DATA GENERATION
# 
# This is the KEY difference from KNN and XGBoost.
# We generate (patient_profile, meal) pairs across
# MULTIPLE patient profiles so the DNN learns that
# the same meal scores differently for different patients.
#
# This is clinically justified: a 500-cal high-carb meal
# is suitable for a low-risk patient but dangerous for
# a medicated diabetic patient.
# ════════════════════════════════════════════════════════

def calculate_meal_relevance(meal_row, macros, bmi, leicester_score):
    """
    Calculates a clinically grounded relevance score (0.0 - 1.0)
    for a specific meal given a specific patient's constraints.

    Scoring logic:
      1. Carb proximity (50%) — how close is this meal to the
         patient's ideal carb target (midpoint of their window)
      2. Protein reward  (30%) — higher protein = better blood
         sugar stability post-meal
      3. Sugar penalty   (20%) — lower sugar = better glycaemic
         control

    Returns 0.0 if meal violates any hard medical constraint.
    """
    # Hard constraint check — if meal fails, score = 0
    if (meal_row['Calories']  > macros['per_meal_calories']  or
        meal_row['Carbs_g']   > macros['per_meal_max_carbs'] or
        meal_row['Carbs_g']   < macros['per_meal_min_carbs'] or
        meal_row['Sugar_g']   > macros['per_meal_sugar']):
        return 0.0

    # Carb proximity score
    carb_target  = (macros['per_meal_max_carbs'] +
                    macros['per_meal_min_carbs']) / 2
    carb_range   = max(macros['per_meal_max_carbs'] -
                       macros['per_meal_min_carbs'], 1)
    carb_score   = 1.0 - (abs(meal_row['Carbs_g'] - carb_target) /
                           carb_range)
    carb_score   = max(0.0, min(1.0, carb_score))

    # Protein reward score
    protein_score = min(1.0, meal_row['Protein_g'] / 50.0)

    # Sugar penalty score (lower sugar = higher score)
    sugar_score  = 1.0 - (meal_row['Sugar_g'] /
                           (macros['per_meal_sugar'] + 1e-9))
    sugar_score  = max(0.0, min(1.0, sugar_score))

    # Weighted combination
    return round(
        carb_score   * 0.50 +
        protein_score * 0.30 +
        sugar_score  * 0.20,
        4
    )


# ── Define diverse synthetic patient profiles ────────────
# Covers the full spectrum of your app's user base
SYNTHETIC_PATIENTS = [
    # (age, weight, height, gender, activity, leicester, medication)
    (25, 60, 165, 'female', 'moderate',  6,  False),  # young low-risk female
    (35, 75, 170, 'male',   'light',     9,  False),  # mid-age low-risk male
    (42, 80, 168, 'female', 'sedentary', 12, False),  # pre-diabetic female
    (45, 85, 175, 'male',   'sedentary', 14, True),   # diabetic medicated male
    (50, 90, 172, 'male',   'sedentary', 16, True),   # diabetic high-risk male
    (55, 70, 160, 'female', 'light',     15, False),  # pre-diabetic older female
    (60, 95, 178, 'male',   'sedentary', 18, True),   # diabetic elderly male
    (38, 65, 162, 'female', 'moderate',  11, False),  # pre-diabetic active female
    (48, 88, 174, 'male',   'light',     17, True),   # diabetic male
    (30, 55, 158, 'female', 'active',    5,  False),  # young active low-risk
]

print('Generating synthetic training pairs...')
print(f'  Patient profiles  : {len(SYNTHETIC_PATIENTS)}')
print(f'  Recipes in dataset: {len(df):,}')
print(f'  Max training pairs: {len(SYNTHETIC_PATIENTS) * len(df):,}')

training_rows = []

for (age, wt, ht, gen, act, leic, med) in SYNTHETIC_PATIENTS:
    tdee   = calculate_tdee(age, wt, ht, gen, act)
    macros = get_dynamic_macros(tdee, leic, med)
    bmi    = round(wt / ((ht / 100) ** 2), 1)

    # Only score meals that are in a reasonable range
    # (not too far outside constraints — DNN still needs
    # both positive AND near-miss examples to learn boundaries)
    candidate_df = df[
        (df['Calories'] <= macros['per_meal_calories'] * 1.2) &
        (df['Carbs_g']  <= macros['per_meal_max_carbs'] * 1.3) &
        (df['Sugar_g']  <= macros['per_meal_sugar'] * 1.5)
    ]

    for _, meal in candidate_df.iterrows():
        score = calculate_meal_relevance(meal, macros, bmi, leic)
        training_rows.append({
            # Patient features
            'p_age'         : age,
            'p_bmi'         : bmi,
            'p_leicester'   : leic,
            'p_tdee'        : tdee,
            'p_medication'  : int(med),
            'p_carb_target' : (macros['per_meal_max_carbs'] +
                               macros['per_meal_min_carbs']) / 2,
            'p_sugar_limit' : macros['per_meal_sugar'],
            'p_cal_limit'   : macros['per_meal_calories'],
            # Meal features
            'm_calories'    : meal['Calories'],
            'm_carbs'       : meal['Carbs_g'],
            'm_sugar'       : meal['Sugar_g'],
            'm_protein'     : meal['Protein_g'],
            'm_fat'         : meal['Fat_g'],
            # Target
            'relevance'     : score
        })

train_df = pd.DataFrame(training_rows)
print(f'\nTraining pairs generated: {len(train_df):,}')
print(f'  Positive pairs (score > 0): '
      f'{(train_df["relevance"] > 0).sum():,}')
print(f'  Zero-score pairs          : '
      f'{(train_df["relevance"] == 0).sum():,}')
print(f'  Avg relevance score       : '
      f'{train_df["relevance"].mean():.4f}')

Generating synthetic training pairs...
  Patient profiles  : 10
  Recipes in dataset: 117,662
  Max training pairs: 1,176,620

Training pairs generated: 984,759
  Positive pairs (score > 0): 569,510
  Zero-score pairs          : 415,249
  Avg relevance score       : 0.3319


In [5]:
# ════════════════════════════════════════════════════════
# CELL 4 — FEATURE PREPARATION & NORMALIZATION
# ════════════════════════════════════════════════════════

PATIENT_FEATURES = [
    'p_age', 'p_bmi', 'p_leicester', 'p_tdee',
    'p_medication', 'p_carb_target', 'p_sugar_limit', 'p_cal_limit'
]

MEAL_FEATURES = [
    'm_calories', 'm_carbs', 'm_sugar', 'm_protein', 'm_fat'
]

ALL_FEATURES = PATIENT_FEATURES + MEAL_FEATURES

X = train_df[ALL_FEATURES].values.astype(np.float32)
y = train_df['relevance'].values.astype(np.float32)

# Normalize all features to 0-1
# Important: fit scaler on training set only
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)

scaler_X = MinMaxScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled  = scaler_X.transform(X_test)

print('Feature preparation complete')
print('-' * 40)
print(f'  Total features   : {len(ALL_FEATURES)}')
print(f'    Patient inputs : {len(PATIENT_FEATURES)}')
print(f'    Meal inputs    : {len(MEAL_FEATURES)}')
print(f'  Training samples : {len(X_train):,}')
print(f'  Test samples     : {len(X_test):,}')
print(f'  Target range     : {y.min():.3f} - {y.max():.3f}')

Feature preparation complete
----------------------------------------
  Total features   : 13
    Patient inputs : 8
    Meal inputs    : 5
  Training samples : 837,045
  Test samples     : 147,714
  Target range     : 0.000 - 0.994


In [6]:
# ════════════════════════════════════════════════════════
# CELL 5 — DNN ARCHITECTURE
#
# Architecture: Two-branch input
#   Branch 1: Patient profile (8 features)
#   Branch 2: Meal nutrition  (5 features)
#   Merged  : Concatenated and passed through deep layers
#   Output  : Single relevance score (0-1)
#
# Design decisions:
#   - Separate branches: forces model to learn patient and
#     meal representations independently before combining
#   - BatchNormalization: stabilizes training on mixed-scale
#     nutritional data
#   - Dropout: prevents overfitting on synthetic labels
#   - L2 regularization: prevents large weights on small
#     feature set
#   - Sigmoid output: constrains score to 0-1 range
# ════════════════════════════════════════════════════════

def build_dnn_model(n_patient_features, n_meal_features,
                    learning_rate=0.001):
    """
    Two-branch DNN for patient-aware meal scoring.
    """
    # ── Patient branch ───────────────────────────────────
    patient_input = keras.Input(shape=(n_patient_features,),
                                name='patient_input')
    p = layers.Dense(64, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4),
                     name='patient_dense_1')(patient_input)
    p = layers.BatchNormalization(name='patient_bn_1')(p)
    p = layers.Dropout(0.2, name='patient_drop_1')(p)
    p = layers.Dense(32, activation='relu',
                     name='patient_dense_2')(p)

    # ── Meal branch ──────────────────────────────────────
    meal_input = keras.Input(shape=(n_meal_features,),
                             name='meal_input')
    m = layers.Dense(64, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4),
                     name='meal_dense_1')(meal_input)
    m = layers.BatchNormalization(name='meal_bn_1')(m)
    m = layers.Dropout(0.2, name='meal_drop_1')(m)
    m = layers.Dense(32, activation='relu',
                     name='meal_dense_2')(m)

    # ── Merge & deep layers ──────────────────────────────
    merged = layers.Concatenate(name='merge')([p, m])  # 64-dim

    x = layers.Dense(128, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4),
                     name='merged_dense_1')(merged)
    x = layers.BatchNormalization(name='merged_bn_1')(x)
    x = layers.Dropout(0.3, name='merged_drop_1')(x)

    x = layers.Dense(64, activation='relu',
                     name='merged_dense_2')(x)
    x = layers.BatchNormalization(name='merged_bn_2')(x)
    x = layers.Dropout(0.2, name='merged_drop_2')(x)

    x = layers.Dense(32, activation='relu',
                     name='merged_dense_3')(x)

    # ── Output ───────────────────────────────────────────
    output = layers.Dense(1, activation='sigmoid',
                          name='relevance_score')(x)

    model = keras.Model(
        inputs=[patient_input, meal_input],
        outputs=output,
        name='Diapilot_DNN'
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',                # regression task
        metrics=['mae']
    )
    return model


dnn_model = build_dnn_model(
    n_patient_features=len(PATIENT_FEATURES),
    n_meal_features=len(MEAL_FEATURES)
)

dnn_model.summary()
total_params = dnn_model.count_params()
print(f'\nTotal trainable parameters: {total_params:,}')

Model: "Diapilot_DNN"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ patient_input       │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ meal_input          │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patient_dense_1     │ (None, 64)        │        576 │ patient_input[0]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ meal_dense_1        │ (None, 64)        │        384 │ meal_input[0][0]  │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patient_bn_1        │ (None, 64)        │        256 │ patient_dense_1[… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ meal_bn_1           │ (None, 64)        │        256 │ meal_dense_1[0][… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patient_drop_1      │ (None, 64)        │          0 │ patient_bn_1[0][… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ meal_drop_1         │ (None, 64)        │          0 │ meal_bn_1[0][0]   │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patient_dense_2     │ (None, 32)        │      2,080 │ patient_drop_1[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ meal_dense_2        │ (None, 32)        │      2,080 │ meal_drop_1[0][0] │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ merge (Concatenate) │ (None, 64)        │          0 │ patient_dense_2[… │
│                     │                   │            │ meal_dense_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ merged_dense_1      │ (None, 128)       │      8,320 │ merge[0][0]       │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ merged_bn_1         │ (None, 128)       │        512 │ merged_dense_1[0… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ merged_drop_1       │ (None, 128)       │          0 │ merged_bn_1[0][0] │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ merged_dense_2      │ (None, 64)        │      8,256 │ merged_drop_1[0]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ merged_bn_2         │ (None, 64)        │        256 │ merged_dense_2[0… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ merged_drop_2       │ (None, 64)        │          0 │ merged_bn_2[0][0

 Total params: 25,089 (98.00 KB)

 Trainable params: 24,449 (95.50 KB)

 Non-trainable params: 640 (2.50 KB)


Total trainable parameters: 25,089


In [7]:
# ════════════════════════════════════════════════════════
# CELL 6 — TRAINING
# ════════════════════════════════════════════════════════

# Split scaled features back into patient and meal branches
n_p = len(PATIENT_FEATURES)
X_train_patient = X_train_scaled[:, :n_p]
X_train_meal    = X_train_scaled[:, n_p:]
X_test_patient  = X_test_scaled[:, :n_p]
X_test_meal     = X_test_scaled[:, n_p:]

# Callbacks
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

lr_scheduler = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=4,
    min_lr=1e-6,
    verbose=1
)

os.makedirs('../models', exist_ok=True)
model_checkpoint = callbacks.ModelCheckpoint(
    '../models/diapilot_dnn_best.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=0
)

print('Training DNN...')
print('-' * 40)

history = dnn_model.fit(
    [X_train_patient, X_train_meal],
    y_train,
    validation_split=0.15,
    epochs=60,
    batch_size=512,
    callbacks=[early_stop, lr_scheduler, model_checkpoint],
    verbose=1
)

print('\nTraining complete.')
print(f'  Best val_loss : {min(history.history["val_loss"]):.6f}')
print(f'  Best val_mae  : {min(history.history["val_mae"]):.6f}')
print(f'  Epochs run    : {len(history.history["loss"])}')

Training DNN...
----------------------------------------
Epoch 1/60
1390/1390 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.0213 - mae: 0.0636 - val_loss: 0.0091 - val_mae: 0.0308 - learning_rate: 0.0010
Epoch 2/60
1390/1390 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0085 - mae: 0.0360 - val_loss: 0.0043 - val_mae: 0.0210 - learning_rate: 0.0010
Epoch 3/60
1390/1390 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0065 - mae: 0.0305 - val_loss: 0.0058 - val_mae: 0.0232 - learning_rate: 0.0010
Epoch 4/60
1390/1390 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0055 - mae: 0.0275 - val_loss: 0.0053 - val_mae: 0.0243 - learning_rate: 0.0010
Epoch 5/60
1390/1390 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0050 - mae: 0.0256 - val_loss: 0.0040 - val_mae: 0.0213 - learning_rate: 0.0010
Epoch 6/60
1390/1390 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0046 - mae: 0.0242 - val_loss: 0.0041 - val_mae: 0.0205 - learning_rate: 0.0010
Epoch 7/60
1390/1390 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0043 - mae: 0.0

In [8]:
# ════════════════════════════════════════════════════════
# CELL 7 — EVALUATION ON TEST SET
# ════════════════════════════════════════════════════════

# Load best checkpoint
best_model = keras.models.load_model('../models/diapilot_dnn_best.keras')

y_pred = best_model.predict(
    [X_test_patient, X_test_meal], verbose=0
).flatten()

mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)
rmse = np.sqrt(np.mean((y_test - y_pred) ** 2))

# Compliance: DNN predicted score > 0.3 = recommended
# Check if recommended meals actually pass medical constraints
threshold     = 0.3
recommended   = y_pred >= threshold
actual_good   = y_test  > 0
precision     = (recommended & actual_good).sum() / recommended.sum()
recall        = (recommended & actual_good).sum() / actual_good.sum()

print('=' * 55)
print('   DNN EVALUATION REPORT')
print('=' * 55)
print(f'\nRegression Metrics:')
print(f'  MAE  (lower=better): {mae:.4f}')
print(f'  RMSE (lower=better): {rmse:.4f}')
print(f'  R2   (higher=better): {r2:.4f}')
print(f'\nRecommendation Quality (threshold={threshold}):')
print(f'  Precision : {precision:.4f}  '
      f'(of meals DNN recommended, how many are clinically valid)')
print(f'  Recall    : {recall:.4f}  '
      f'(of valid meals, how many did DNN find)')
print(f'\nScore Distribution (predicted):')
print(f'  Mean  : {y_pred.mean():.4f}')
print(f'  Std   : {y_pred.std():.4f}')
print(f'  Min   : {y_pred.min():.4f}')
print(f'  Max   : {y_pred.max():.4f}')
print('=' * 55)

   DNN EVALUATION REPORT

Regression Metrics:
  MAE  (lower=better): 0.0157
  RMSE (lower=better): 0.0419
  R2   (higher=better): 0.9797

Recommendation Quality (threshold=0.3):
  Precision : 0.9930  (of meals DNN recommended, how many are clinically valid)
  Recall    : 0.9992  (of valid meals, how many did DNN find)

Score Distribution (predicted):
  Mean  : 0.3284
  Std   : 0.2835
  Min   : 0.0000
  Max   : 0.8321


In [9]:
# ════════════════════════════════════════════════════════
# CELL 8 — MEAL TYPE CLASSIFICATION
# (same as KNN & XGBoost for fair comparison)
# ════════════════════════════════════════════════════════

BREAKFAST_KW = ['egg','oat','oatmeal','paratha','halwa','dahi','yogurt',
                'pancake','toast','cereal','porridge','smoothie','muffin',
                'waffle','granola','scramble','omelette','crepe']
DINNER_KW    = ['karahi','biryani','gosht','daal','curry','sabzi','stew',
                'roast','steak','bake','casserole','kebab','tikka','nihari',
                'korma','palak','keema','chops','grilled chicken','lamb']

def classify_meal_type(row):
    text    = (str(row['Recipe_Name']) + ' ' + str(row['Ingredients'])).lower()
    b_score = sum(1 for kw in BREAKFAST_KW if kw in text)
    d_score = sum(1 for kw in DINNER_KW   if kw in text)
    if b_score > d_score: return 'Breakfast'
    elif d_score > 0:     return 'Dinner'
    return 'Lunch'

df['Meal_Type'] = df.apply(classify_meal_type, axis=1)
counts = df['Meal_Type'].value_counts()
print('Meal type classification complete')
for mt in ['Breakfast','Lunch','Dinner']:
    print(f'  {mt:12s}: {counts.get(mt,0):,} recipes')

Meal type classification complete
  Breakfast   : 28,429 recipes
  Lunch       : 63,925 recipes
  Dinner      : 25,308 recipes


In [10]:
# ════════════════════════════════════════════════════════
# CELL 9 — INFERENCE: SCORE ALL SAFE MEALS FOR THIS PATIENT
# ════════════════════════════════════════════════════════

# Step 1: Apply medical filter
safe_df = df[
    (df['Calories'] <= patient_macros['per_meal_calories']) &
    (df['Carbs_g']  <= patient_macros['per_meal_max_carbs']) &
    (df['Carbs_g']  >= patient_macros['per_meal_min_carbs']) &
    (df['Sugar_g']  <= patient_macros['per_meal_sugar'])
].copy().reset_index(drop=True)

print(f'Medically safe pool: {len(safe_df):,} recipes')

# Step 2: Build inference feature matrix
# Patient features are the same for every row (this patient)
patient_vector = np.array([[
    PATIENT_AGE,
    PATIENT_WEIGHT_KG / ((PATIENT_HEIGHT_CM / 100) ** 2),  # BMI
    LEICESTER_SCORE,
    patient_tdee,
    int(ON_MEDICATION),
    (patient_macros['per_meal_max_carbs'] +
     patient_macros['per_meal_min_carbs']) / 2,
    patient_macros['per_meal_sugar'],
    patient_macros['per_meal_calories']
]])

# Repeat patient vector for every meal in safe pool
patient_matrix = np.repeat(patient_vector, len(safe_df), axis=0)

# Meal features
meal_matrix = safe_df[['Calories','Carbs_g','Sugar_g',
                         'Protein_g','Fat_g']].values

# Combine and scale using the SAME scaler fitted on training data
combined     = np.hstack([patient_matrix, meal_matrix]).astype(np.float32)
combined_scaled = scaler_X.transform(combined)

inf_patient = combined_scaled[:, :n_p]
inf_meal    = combined_scaled[:, n_p:]

# Step 3: Predict relevance scores
print('Running DNN inference on safe meal pool...')
safe_df['DNN_Score'] = best_model.predict(
    [inf_patient, inf_meal],
    batch_size=1024,
    verbose=0
).flatten()

print(f'Inference complete.')
print(f'  Score range: {safe_df["DNN_Score"].min():.4f}'
      f' - {safe_df["DNN_Score"].max():.4f}')
print(f'  Avg score  : {safe_df["DNN_Score"].mean():.4f}')

Medically safe pool: 21,763 recipes
Running DNN inference on safe meal pool...
Inference complete.
  Score range: 0.2702 - 0.8001
  Avg score  : 0.5510


In [11]:
# ════════════════════════════════════════════════════════
# CELL 10 — MMR DIVERSITY SELECTION
# (same algorithm as KNN & XGBoost for fair comparison)
# ════════════════════════════════════════════════════════

def apply_mmr_dnn(candidate_df, top_k, lambda_param=0.7):
    """
    MMR using DNN_Score as relevance.
    Diversity measured across 4 macros.
    """
    if len(candidate_df) == 0:
        return pd.DataFrame()
    if len(candidate_df) <= top_k:
        return candidate_df.reset_index(drop=True)

    macro_cols = ['Calories','Carbs_g','Sugar_g','Protein_g']
    macro_vals = candidate_df[macro_cols].values.astype(float)
    col_range  = macro_vals.max(axis=0) - macro_vals.min(axis=0)
    col_range  = np.where(col_range > 0, col_range, 1.0)
    macro_norm = (macro_vals - macro_vals.min(axis=0)) / col_range

    scores    = candidate_df['DNN_Score'].values
    remaining = list(range(len(candidate_df)))
    selected  = []

    # Start with highest DNN score
    best_start = int(np.argmax(scores))
    selected.append(best_start)
    remaining.remove(best_start)

    while len(selected) < top_k and remaining:
        best_mmr = -float('inf')
        best_idx = None
        for i in remaining:
            relevance = scores[i]
            sims      = [
                1.0 / (1.0 + np.linalg.norm(
                    macro_norm[i] - macro_norm[s]))
                for s in selected
            ]
            mmr = lambda_param * relevance - \
                  (1 - lambda_param) * max(sims)
            if mmr > best_mmr:
                best_mmr = mmr
                best_idx = i
        selected.append(best_idx)
        remaining.remove(best_idx)

    return candidate_df.iloc[selected].reset_index(drop=True)


print('MMR function ready.')

MMR function ready.


In [12]:
# ════════════════════════════════════════════════════════
# CELL 11 — GENERATE 30-DAY PLAN
# ════════════════════════════════════════════════════════

DAYS_IN_PLAN = 30
MEAL_SLOTS   = ['Breakfast', 'Lunch', 'Dinner']

plan_rows    = []
pool_summary = {}

for meal_type in MEAL_SLOTS:
    pool = safe_df[safe_df['Meal_Type'] == meal_type].copy()

    if len(pool) < 10:
        print(f'  WARNING: {meal_type} only {len(pool)} recipes'
              f' — using full safe pool')
        pool = safe_df.copy()

    selected         = apply_mmr_dnn(pool, top_k=DAYS_IN_PLAN)
    pool_summary[meal_type] = len(selected)

    for day_idx in range(DAYS_IN_PLAN):
        row = selected.iloc[day_idx % len(selected)]
        plan_rows.append({
            'Day'         : f'Day {day_idx + 1}',
            'Day_Num'     : day_idx + 1,
            'Meal'        : meal_type,
            'Recipe'      : row['Recipe_Name'],
            'Ingredients' : str(row['Ingredients']),
            'Calories'    : round(row['Calories'],  1),
            'Carbs_g'     : round(row['Carbs_g'],   1),
            'Sugar_g'     : round(row['Sugar_g'],   1),
            'Protein_g'   : round(row['Protein_g'], 1),
        })

meal_order     = {'Breakfast':0,'Lunch':1,'Dinner':2}
plan_df        = pd.DataFrame(plan_rows)
plan_df['Meal_Order'] = plan_df['Meal'].map(meal_order)
plan_df        = (
    plan_df
    .sort_values(['Day_Num','Meal_Order'])
    .drop(columns=['Day_Num','Meal_Order'])
    .reset_index(drop=True)
)

print('30-Day DNN Diet Plan Generated')
print('=' * 55)
print(f'  Total meals : {len(plan_df)} (30 days x 3 slots)')
for mt, c in pool_summary.items():
    print(f'  {mt:12s}: {c} unique meals by MMR')
print()
print('Columns in plan:', list(plan_df.columns))
print()
print('Sample — First 3 meals:')
print(plan_df[['Day','Meal','Recipe','Calories','Carbs_g','Sugar_g']].head(3).to_string(index=False))


30-Day DNN Diet Plan Generated
  Total meals : 90 (30 days x 3 slots)
  Breakfast   : 30 unique meals by MMR
  Lunch       : 30 unique meals by MMR
  Dinner      : 30 unique meals by MMR

Columns in plan: ['Day', 'Meal', 'Recipe', 'Ingredients', 'Calories', 'Carbs_g', 'Sugar_g', 'Protein_g']

Sample — First 3 meals:
  Day      Meal                            Recipe  Calories  Carbs_g  Sugar_g
Day 1 Breakfast   linda s new england fried clams     674.5     41.2      0.5
Day 1     Lunch        boston battered fried fish     483.4     49.5      0.5
Day 1    Dinner hot roast beef sandwiches   gravy     666.6     44.0      2.5


In [13]:
# ════════════════════════════════════════════════════════
# CELL 12 — PLAN EVALUATION
# ════════════════════════════════════════════════════════

print('=' * 55)
print('   DNN PLAN EVALUATION')
print('=' * 55)

compliant = plan_df[
    (plan_df['Carbs_g']  <= patient_macros['per_meal_max_carbs']) &
    (plan_df['Carbs_g']  >= patient_macros['per_meal_min_carbs']) &
    (plan_df['Sugar_g']  <= patient_macros['per_meal_sugar'])     &
    (plan_df['Calories'] <= patient_macros['per_meal_calories'])
]

print(f'\n1. Medical Compliance')
print(f'   ADA-safe meals  : {len(compliant)} / {len(plan_df)}')
print(f'   Rate            : {len(compliant)/len(plan_df)*100:.1f}%')

print(f'\n2. Nutritional Profile')
print(f'   Avg Calories    : {plan_df["Calories"].mean():.1f} kcal')
print(f'   Avg Carbs       : {plan_df["Carbs_g"].mean():.1f}g (ADA target 45-60g)')
print(f'   Avg Sugar       : {plan_df["Sugar_g"].mean():.1f}g')
print(f'   Avg Protein     : {plan_df["Protein_g"].mean():.1f}g')

unique = plan_df['Recipe'].nunique()
print(f'\n3. Variety')
print(f'   Unique recipes  : {unique} / {len(plan_df)}')
print(f'   Variety score   : {unique/len(plan_df)*100:.1f}%')

print(f'\n4. Per-Slot Variety')
for mt in MEAL_SLOTS:
    u = plan_df[plan_df['Meal']==mt]['Recipe'].nunique()
    print(f'   {mt:12s}: {u} unique / 30 days')

in_ada = plan_df[
    (plan_df['Carbs_g'] >= 45) & (plan_df['Carbs_g'] <= 60)
]
print(f'\n5. ADA Carb Range (45-60g)')
print(f'   In range: {len(in_ada)} / {len(plan_df)} meals')
print('\n' + '=' * 55)


   DNN PLAN EVALUATION

1. Medical Compliance
   ADA-safe meals  : 90 / 90
   Rate            : 100.0%

2. Nutritional Profile
   Avg Calories    : 557.3 kcal
   Avg Carbs       : 42.8g (ADA target 45-60g)
   Avg Sugar       : 3.1g
   Avg Protein     : 51.9g

3. Variety
   Unique recipes  : 90 / 90
   Variety score   : 100.0%

4. Per-Slot Variety
   Breakfast   : 30 unique / 30 days
   Lunch       : 30 unique / 30 days
   Dinner      : 30 unique / 30 days

5. ADA Carb Range (45-60g)
   In range: 32 / 90 meals



In [14]:
# ════════════════════════════════════════════════════════
# CELL 13 — SAVE CSV (final output)
# ════════════════════════════════════════════════════════

import os

os.makedirs('../reports', exist_ok=True)

csv_path = '../reports/Diapilot_DNN_Diet_Plan.csv'
plan_df.to_csv(csv_path, index=False)

print('CSV saved: ' + csv_path)
print('Total meals: ' + str(len(plan_df)))
print()
print('All columns:')
for col in plan_df.columns:
    sample = str(plan_df[col].iloc[0])[:50]
    print(f'  {col:15s}: {sample}')


CSV saved: ../reports/Diapilot_DNN_Diet_Plan.csv
Total meals: 90

All columns:
  Day            : Day 1
  Meal           : Breakfast
  Recipe         : linda s new england fried clams
  Ingredients    : ['clams', 'vegetable oil', 'eggs', 'evaporated mil
  Calories       : 674.5
  Carbs_g        : 41.2
  Sugar_g        : 0.5
  Protein_g      : 69.5


In [15]:
%pip install fpdf

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\SAAD\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
